In [1]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.manifold import MDS
from sklearn.decomposition import PCA

from matplotlib import pyplot as plt

# --- Zufällige Datenmatrix ---
np.random.seed(42)
X = np.random.rand(42, 10).T

# --- zscore ---
Z = (X - X.mean(axis=0)) / X.std(axis=0)
# Alternativ: Nicht normierte Daten
# Z = X

# --- Abstände aller Datenpunkte zueinander ---
M = cdist(Z, Z)

# --- Quadratische Abstädnde ---
D = M**2

# --- Doppelzentrierung ---
N = M.shape[0]
H = np.eye(N) - 1/N * np.ones((N, N))

B = -1/2 * H @ D @ H

# --- Eigenwertzerlegung ---
eig_vals, eig_vecs = np.linalg.eigh(B)

# --- Eigenvektoren nach Eigenwerten sortieren ---
idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:,idx]

# --- Proportion of Distance Matrix Explained -> PDME ---
PDME = eig_vals / np.abs(eig_vals).sum()

# --- Visualisierung ---
# plt.plot(range(1, PDME.shape[0]+1), PDME)
# plt.xlabel("MDS-Komponente")
# plt.ylabel("PDME")
# plt.show()

# plt.plot(range(1, PDME.shape[0]+1), PDME.cumsum())
# plt.xlabel("MDS-Komponente")
# plt.ylabel("PDME")
# plt.show()

# --- Eliminierung negativer Eigenwerte ---
idx = eig_vals > 0
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:, idx]
PDME = PDME[idx]

# --- Alle MDS-Komponenten verwenden, die mehr als 10% der Varianz ausmachen ---
# Alternativ: Verwende alle MDS-Komponenten zur (Wieder)-Herstellung von X
idx = PDME > 0.1
# Alternativ: Verwende 2 MDS-Komponenten zur Visualisierung
# idx = range(2)
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:,idx]

# --- Projizieren der Daten auf die MDS-Komponenten -> C ---
C = eig_vecs @ np.sqrt(np.diag(eig_vals))

# --- Vergleichswert ---
# Obacht: Vorzeichen der Eigenvektoren nicht eindeutig
# mds = MDS(n_components=eig_vecs.shape[1], eps=0, normalized_stress="auto")
# C_ = mds.fit_transform(Z)
# Alternativ: Wenn X zentriert ist und D aus eukl. Abständen besteht, MDS ~ PCA
# pca = PCA(n_components=eig_vecs.shape[1])
# C_ = pca.fit_transform(Z)
# Alternativ: Verwende M
# mds = MDS(n_components=eig_vecs.shape[1], eps=0, dissimilarity="precomputed", normalized_stress="auto")
# C_ = mds.fit_transform(M)

# --- Visualisierung ---
# assert C.shape[1] == 2, f"{C.shape=}"
# plt.scatter(*C.T, label="spicker")
# plt.scatter(*C_.T, marker="x", label="sklearn")
# plt.legend()
# plt.xlabel("1. MDS-Komponente")
# plt.ylabel("2. MDS-Komponente")
# plt.show()